In [1]:
import pandas as pd

df = pd.read_csv("/content/telco_customer_churn_cleaned.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,TenureGroup,MonthlyRevenue,RevenuePerTenureMonth
0,7590-VHVEG,Female,No,Yes,No,1,No,No phone service,DSL,No,...,No,Month-to-month,Yes,Electronic check,29.85,29.85,0,0-6 Months,29.85,29.850000
1,5575-GNVDE,Male,No,No,No,34,Yes,No,DSL,Yes,...,No,One year,No,Mailed check,56.95,1889.50,0,25-48 Months,56.95,55.573529
2,3668-QPYBK,Male,No,No,No,2,Yes,No,DSL,Yes,...,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,0-6 Months,53.85,54.075000
3,7795-CFOCW,Male,No,No,No,45,No,No phone service,DSL,Yes,...,No,One year,No,Bank transfer (automatic),42.30,1840.75,0,25-48 Months,42.30,40.905556
4,9237-HQITU,Female,No,No,No,2,Yes,No,Fiber optic,No,...,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,0-6 Months,70.70,75.825000


In [2]:
df["ValueSegment"] = pd.qcut(
    df["MonthlyCharges"],
    q=3,
    labels=["Low Value", "Medium Value", "High Value"]
)

In [3]:
df.groupby("ValueSegment", observed=True).agg(
    Customers=("customerID", "count"),
    AvgMonthlyCharges=("MonthlyCharges", "mean"),
    AvgTotalCharges=("TotalCharges", "mean"),
    ChurnRate=("Churn", "mean")
).round(2)

,Customers,AvgMonthlyCharges,AvgTotalCharges,ChurnRate
ValueSegment,,,,
Low Value,2351,27.99,715.37,0.16
Medium Value,2345,69.04,1954.69,0.30
High Value,2347,97.33,4171.53,0.34


In [4]:
value_summary = df.groupby("ValueSegment", observed=True).agg(
    Customers=("customerID", "count"),
    AvgMonthlyCharges=("MonthlyCharges", "mean"),
    AvgTotalCharges=("TotalCharges", "mean"),
    ChurnRate=("Churn", "mean")
)

value_summary["ChurnRate"] = value_summary["ChurnRate"] * 100

value_summary.round(2)

,Customers,AvgMonthlyCharges,AvgTotalCharges,ChurnRate
ValueSegment,,,,
Low Value,2351,27.99,715.37,15.87
Medium Value,2345,69.04,1954.69,29.68
High Value,2347,97.33,4171.53,34.09


In [5]:
df["TenureGroup"].value_counts().sort_index()

,count
TenureGroup,
0-6 Months,1481
13-24 Months,1024
25-48 Months,1594
49-72 Months,2239
7-12 Months,705


In [6]:
segment_summary = df.groupby(
    ["TenureGroup", "ValueSegment"],
    observed=True
).agg(
    Customers=("customerID", "count"),
    ChurnRate=("Churn", "mean"),
    MonthlyRevenue=("MonthlyCharges", "sum")
)

segment_summary["ChurnRate"] = segment_summary["ChurnRate"] * 100

segment_summary = segment_summary.round(2)

segment_summary

Customers  ChurnRate  MonthlyRevenue
TenureGroup  ValueSegment                                      
0-6 Months   Low Value           676      36.69        20282.75
             Medium Value        590      62.71        41068.25
             High Value          215      77.21        19716.95
13-24 Months Low Value           361      10.53         9967.25
             Medium Value        369      28.46        25297.95
             High Value          294      51.36        27564.65
25-48 Months Low Value           492       5.49        13372.85
             Medium Value        521      17.85        35619.45
             High Value          581      35.28        56101.00
49-72 Months Low Value           549       2.55        14444.05
             Medium Value        595       5.55        41192.55
             High Value         1095      15.16       109927.10
7-12 Months  Low Value           273      16.85         7734.35
             Medium Value        270      35.19        18712.00
             High Value          162      69.14        15115.45

In [7]:
segment_summary.sort_values(
    "ChurnRate",
    ascending=False
)

,,Customers,ChurnRate,MonthlyRevenue
TenureGroup,ValueSegment,,,
0-6 Months,High Value,215,77.21,19716.95
7-12 Months,High Value,162,69.14,15115.45
0-6 Months,Medium Value,590,62.71,41068.25
13-24 Months,High Value,294,51.36,27564.65
0-6 Months,Low Value,676,36.69,20282.75
25-48 Months,High Value,581,35.28,56101.00
7-12 Months,Medium Value,270,35.19,18712.00
13-24 Months,Medium Value,369,28.46,25297.95
25-48 Months,Medium Value,521,17.85,35619.45


# CHURNED REVENUE

In [8]:
churned_revenue = df.loc[
    df["Churn"] == 1,
    "MonthlyCharges"
].sum()

churned_revenue

np.float64(139130.85)

In [11]:
total_revenue = df["MonthlyCharges"].sum()

total_revenue
churned_revenue / total_revenue * 100

np.float64(30.503351555282137)

# HIGH VALUE CHURN

In [12]:
high_value_churn = df[
    (df["ValueSegment"] == "High Value") &
    (df["Churn"] == 1)
]

high_value_churn.shape
high_value_churn["MonthlyCharges"].sum()

np.float64(76721.5)

# FINAL

In [13]:
value_summary.round(2)

,Customers,AvgMonthlyCharges,AvgTotalCharges,ChurnRate
ValueSegment,,,,
Low Value,2351,27.99,715.37,15.87
Medium Value,2345,69.04,1954.69,29.68
High Value,2347,97.33,4171.53,34.09


In [14]:
segment_summary.sort_values("ChurnRate", ascending=False)

,,Customers,ChurnRate,MonthlyRevenue
TenureGroup,ValueSegment,,,
0-6 Months,High Value,215,77.21,19716.95
7-12 Months,High Value,162,69.14,15115.45
0-6 Months,Medium Value,590,62.71,41068.25
13-24 Months,High Value,294,51.36,27564.65
0-6 Months,Low Value,676,36.69,20282.75
25-48 Months,High Value,581,35.28,56101.00
7-12 Months,Medium Value,270,35.19,18712.00
13-24 Months,Medium Value,369,28.46,25297.95
25-48 Months,Medium Value,521,17.85,35619.45


Insight 1

Overall churn is 26.54%.

Insight 2

Churn increases substantially with customer value, reaching 34.09% among high-value customers versus 15.87% among low-value customers.

Insight 3

Early-tenure high-value customers are the highest-risk segment, with churn of 77.21% in the first six months.

And financially:

Insight 4

Customers who churned represent ₹139.1K in monthly charges, with ₹76.7K coming from high-value customers.